This notebook covers experiments with different models and evaluation of their effectiveness.
The following approaches will be checked:
- **Historical Averages**
    - type: baseline
    - reason: the historical average will be evaluated to provide a minimal accuracy level expected from the model
- **KNN**
    - type: baseline
    - reason: based on the EDA results, no linearity between the models was revealed, so knn will be evaluated as an alternative approach to the historical averages, as it essentially finds the closest data points and computes their averages.
- **Poisson Regression**
    - type: baseline
    - reason: this model type perfectly fits the regression problem because of the sold_quantity as a countable value.
- **CatBoost - 2 stages: Classification and Regression**
    - type: advanced
    - reason: because of the zero-inflated dataset, a classification approach to predict if there will be a sale potentially (yes/no) will help a regression model to better understand where 0 value is expected from the beginning.

In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
import numpy as np
pd.set_option('display.max_rows', None)

In [2]:
df = pd.read_csv("../interim_files/notebooks/complete_data_with_features.csv", parse_dates=['date'])
df.head(1)

,Unnamed: 0,flight_key,flight_number,origin,destination,date,month_name,year,weekday_name,is_weekend,...,hist_avg_l4,hist_count_l4,hist_avg_l5,hist_count_l5,hist_avg_per_pax,hist_avg,hist_level_used,fold_num,item_trend_with_threshold,trend_signal
0,25633,77545a68a2f7e9fd44dd05703926b209,AB020,city_001,city_010,2025-11-15,November,2025.0,Saturday,True,...,0.600571,701,0.566809,2103,0.003557,0.416667,1.0,1,1.0,STABLE


## Filtering out all non selling items

There are items with 0 sales and around 0 (less than 10 sales per year).

All models will eventually predict 0 sales for them and to prevent the conservative behavior such products are excluded from the dataset.

For them only a historical average will be computed.

In [3]:
item_sales = df[['item_id', 'sold_quantity']].groupby('item_id').agg('sum').sort_values(by='sold_quantity')
low_selling_df = item_sales[item_sales['sold_quantity'] == 0].reset_index()
low_selling_items = low_selling_df['item_id'].tolist()
print(low_selling_items)

[]


In [4]:
df = df[~df['item_id'].isin(low_selling_items)].copy()

## Train / test df preparation
The last 2 weeks of DF are reserved to the test set.

In [5]:
test_cutoff = df['date'].max() - pd.Timedelta(weeks=2)
print(f"Train / test split by date: {test_cutoff}")
train_df = df[df['date'] <= test_cutoff]
test_df = df[df['date'] > test_cutoff]
print(f"Train DF size: {train_df.shape}")
print(f"Test DF size: {test_df.shape}")

Train / test split by date: 2026-02-14 00:00:00
Train DF size: (45930, 44)
Test DF size: (6670, 44)


Model's accuracy will be evaluated by 2 different approaches:
- technical:
    - MAE - mean absolute error
    - RMSE - root mean square error
- business - by comparing the predicted value vs the factual ones:
    - accurate_predictions: when an actual value equals to the predicted one
    - potential_waste: when an actual value is lower than the predicted one
    - potential_lost_sale: when an actual value is higher than the predicted one
    - accuracy_score: partial-credit score for near misses, computed as `min(actual, predicted) / max(actual, predicted)` (1.0 when both are 0), so an off-by-one prediction isn't scored the same as a wildly wrong one

In [6]:
# technical metrics evaluation
def evaluate_technical_metrics(y_true, y_predicted):

    mae = mean_absolute_error(y_true, y_predicted)
    rmse = np.sqrt((y_true - y_predicted) ** 2).mean()

    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")

In [8]:
# business metrics evaluation
def evaluate_business_metrics(y_true, y_predicted):
    diff = y_true - y_predicted

    # Relative accuracy for wastage: actual / predicted (closer to 1.0 is better)
    waste_mask = diff < 0
    if waste_mask.sum() > 0:
        waste_rel_acc = (y_true[waste_mask] / y_predicted[waste_mask]).mean()
    else:
        waste_rel_acc = 0.0

    # Relative accuracy for missed sales: predicted / actual (closer to 1.0 is better)
    missed_mask = diff > 0
    if missed_mask.sum() > 0:
        missed_rel_acc = (y_predicted[missed_mask] / y_true[missed_mask]).mean()
    else:
        missed_rel_acc = 0.0

    results = pd.DataFrame({
        "category": ["accurate_prediction", "potential_waste", "potential_lost_sale"],
        "number of records": [
            (diff == 0).sum(),
            waste_mask.sum(),
            missed_mask.sum()
        ],
        "share": [
            round((diff == 0).mean() * 100, 1),
            round(waste_mask.mean() * 100, 1),
            round(missed_mask.mean() * 100, 1)
        ],
        "avg deviation": [
            0,
            diff[waste_mask].mean() if waste_mask.sum() > 0 else 0,
            diff[missed_mask].mean() if missed_mask.sum() > 0 else 0
        ],
        "relative accuracy": [
            1.0,
            waste_rel_acc,
            missed_rel_acc
        ]
    })
    print(results.to_string(index=False))

    # partial-credit accuracy score: min(actual, predicted) / max(actual, predicted), 1.0 when both are 0
    max_val = np.maximum(y_true, y_predicted)
    min_val = np.minimum(y_true, y_predicted)
    safe_max = max_val.where(max_val != 0, 1)
    accuracy_score = (min_val / safe_max).where(max_val != 0, 1.0).mean()
    print(f"\naccuracy_score (partial credit): {accuracy_score * 100:.1f}%")

## Historical averages

The historical averages are computed based on a hierarchical approach depending on the number of available records for a given level:
- **Level 1**: `item × route × sales_hour_bin × duration_bin`
- **Level 2**: `item × route × duration_bin`
- **Level 3**: `item × sales_hour_bin × duration_bin`
- **Level 4**: `item` (all records)
- **Level 5**: `category` (fallback for rare products)

There should be at least 5 data points available in a group to compute an average for a given combination.

In [9]:
test_df = test_df.copy()
test_df['pred_baseline'] = test_df['hist_avg'].round().clip(lower=0).astype(int)

Checking the baseline model's accuracy

In [10]:
y_true = test_df["sold_quantity"]
y_predicted = test_df["pred_baseline"]
evaluate_technical_metrics(y_true, y_predicted)
evaluate_business_metrics(y_true, y_predicted)

MAE:  0.482
RMSE: 0.482
           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               4168   62.5       0.000000           1.000000
    potential_waste               1454   21.8      -1.290922           0.065787
potential_lost_sale               1048   15.7       1.279580           0.183797

accuracy_score (partial credit): 66.8%


## KNN - K Nearest Neighbours

The main reason to try KNN is to narrow the specificity for the data points. Compared to the averages by hierarchical grouping, KNN allows to focus on the nearest data points (3, 5, 7 will be tried out), and then it computes the average. The main difference in the approach will be changing dynamically the threshold, so that the model can be adapted based on the business needs: to prevent wastage or to minimize lost sales.

The following feature groups will be used:
- ordinal categorical: `pax_bin`, `sales_hour_bin`, `duration_bin`, `price_bin`
- nominal categorical: `item_id`, `category`, `is_outbound`, `am_pm`, `route`
- numerical: `flight_duration_hours`, `number_of_passengers`, `hist_avg`, `hist_avg_per_pax`, `hist_level_used`

The modelling process will include the following steps:
1. Categorical features preparation: ordinal encoding + target encoding for the categorical features
2. Numerical and encoded categorical features scaling (KNN is sensitive to the scale)
3. Building the model with different hyperparameters: `n_neighbors = 3, 5, 7`
4. Trying out different thresholds

In [25]:
# listing necessary categories
ordinary_categories = ['pax_bin', 'sales_hour_bin', 'duration_bin', 'price_bin', 'trend_signal']
nominal_categories = ["item_id", "category", "is_outbound", "am_pm", "route"]
cat_features = ordinary_categories + nominal_categories
numerical_features = ['flight_duration_hours', 'number_of_passengers', 'hist_avg', 'hist_avg_per_pax', 'hist_level_used', 'item_trend_with_threshold']
all_categories = ordinary_categories + nominal_categories + numerical_features

In [26]:
# checking NaN values before KNN
print(df[all_categories].isna().sum())

pax_bin                      0
sales_hour_bin               0
duration_bin                 0
price_bin                    0
trend_signal                 0
item_id                      0
category                     0
is_outbound                  0
am_pm                        0
route                        0
flight_duration_hours        0
number_of_passengers         0
hist_avg                     0
hist_avg_per_pax             0
hist_level_used              0
item_trend_with_threshold    0
dtype: int64


In [28]:
# categorical features processing
from category_encoders import TargetEncoder
from sklearn.preprocessing import OrdinalEncoder

ordinary_features = [
    train_df['pax_bin'].unique().tolist(),
    train_df['sales_hour_bin'].unique().tolist(),
    train_df['duration_bin'].unique().tolist(),
    train_df['price_bin'].unique().tolist(),
    train_df['trend_signal'].unique().tolist()
]

oe = OrdinalEncoder(categories=ordinary_features)
ord_encoded_cols = [col + '_enc' for col in ordinary_categories]
train_df[ord_encoded_cols] = oe.fit_transform(train_df[ordinary_categories])
test_df[ord_encoded_cols] = oe.transform(test_df[ordinary_categories])

# target encoding
te = TargetEncoder(cols=nominal_categories)
nom_encoded_cols = [col + '_enc' for col in nominal_categories]
train_df[nom_encoded_cols] = te.fit_transform(train_df[nominal_categories], train_df["sold_quantity"])
test_df[nom_encoded_cols] = te.transform(test_df[nominal_categories], test_df["sold_quantity"])

In [29]:
# numerical features scaling
from sklearn.preprocessing import StandardScaler

features = ord_encoded_cols + nom_encoded_cols + numerical_features

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[features])
X_test = scaler.transform(test_df[features])

y_train = train_df["sold_quantity"]
y_test = test_df["sold_quantity"]

In [30]:
# training and predicting
from sklearn.neighbors import KNeighborsRegressor

results = {}

for k in [3, 5, 7]:
    knn = KNeighborsRegressor(n_neighbors=k, metric="euclidean")
    knn.fit(X_train, y_train)
    results[k] = knn.predict(X_test)

In [31]:
# tuning threshold
import numpy as np

def apply_threshold(raw_pred, threshold=0.5):
    floored = np.floor(raw_pred)
    remainder = raw_pred - floored
    return (floored + (remainder >= threshold).astype(int)).astype(int).clip(min=0)

for k, raw_pred in results.items():
    for t in [0.3, 0.5, 0.7]:
        pred = apply_threshold(raw_pred, threshold=t)
        mae = mean_absolute_error(y_test, pred)
        print(f"\nk={k}, threshold={t:.1f} → MAE={mae:.3f}\n")
        evaluate_business_metrics(y_test, pred)


k=3, threshold=0.3 → MAE=0.724

           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               2895   43.4       0.000000           1.000000
    potential_waste               3020   45.3      -1.265563           0.052687
potential_lost_sale                755   11.3       1.335099           0.225807

accuracy_score (partial credit): 48.3%

k=3, threshold=0.5 → MAE=0.529

           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               4049   60.7       0.000000           1.000000
    potential_waste               1479   22.2      -1.363083           0.072552
potential_lost_sale               1142   17.1       1.322242           0.138792

accuracy_score (partial credit): 64.7%

k=3, threshold=0.7 → MAE=0.462

           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               4452   66.7       0.000000           1.000000
    potential_waste  

## CatBoost — Classification + Regression

In [32]:
# feature preparation
cat_features = nominal_categories + ordinary_categories
num_features = numerical_features
features = cat_features + num_features

print(features)

['item_id', 'category', 'is_outbound', 'am_pm', 'route', 'pax_bin', 'sales_hour_bin', 'duration_bin', 'price_bin', 'trend_signal', 'flight_duration_hours', 'number_of_passengers', 'hist_avg', 'hist_avg_per_pax', 'hist_level_used', 'item_trend_with_threshold']


In [33]:
# classification + regression
train_df["target_cls"] = (train_df["sold_quantity"] > 0).astype(int)
X_train = train_df[features]
X_test = test_df[features]

y_train_cls = train_df["target_cls"]
y_train_reg = train_df[train_df["target_cls"] == 1]["sold_quantity"]
X_train_reg = train_df[train_df["target_cls"] == 1][features]

In [34]:
# parameters tuning
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.model_selection import ParameterGrid

param_grid = {
    'iterations': [300, 500, 700],
    'learning_rate': [0.03, 0.05, 0.1],
    'depth': [4, 6, 8],
}

In [35]:
val_cutoff = train_df['date'].max() - pd.Timedelta(weeks=2)
tune_train = train_df[train_df['date'] <= val_cutoff]
tune_val   = train_df[train_df['date'] > val_cutoff]

tune_train_cls = (tune_train['sold_quantity'] > 0).astype(int)
tune_val_cls   = (tune_val['sold_quantity'] > 0).astype(int)

cls_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features,
    eval_metric='F1',
    random_seed=42,
    verbose=100
)

cls_model.fit(X_train, y_train_cls)

0:	learn: 0.5641875	total: 16.7ms	remaining: 8.35s
100:	learn: 0.5916129	total: 1.88s	remaining: 7.43s
200:	learn: 0.5984764	total: 3.58s	remaining: 5.33s
300:	learn: 0.6016447	total: 5.37s	remaining: 3.55s
400:	learn: 0.6058397	total: 7.29s	remaining: 1.8s
499:	learn: 0.6095176	total: 9.12s	remaining: 0us


CatBoostClassifier(cat_features=['item_id', 'category', 'is_outbound', 'am_pm', 'route', 'pax_bin', 'sales_hour_bin', 'duration_bin', 'price_bin', 'trend_signal'], depth=6, eval_metric='F1', iterations=500, learning_rate=0.05, random_seed=42, verbose=100)

In [36]:
# regressor
reg_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features,
    eval_metric='MAE',
    random_seed=42,
    verbose=100
)
reg_model.fit(X_train_reg, y_train_reg)

0:	learn: 0.9872609	total: 6.56ms	remaining: 3.27s
100:	learn: 0.6605807	total: 632ms	remaining: 2.5s
200:	learn: 0.6466020	total: 1.25s	remaining: 1.85s
300:	learn: 0.6392442	total: 1.83s	remaining: 1.21s
400:	learn: 0.6329405	total: 2.43s	remaining: 601ms
499:	learn: 0.6280180	total: 3.05s	remaining: 0us


CatBoostRegressor(cat_features=['item_id', 'category', 'is_outbound', 'am_pm', 'route', 'pax_bin', 'sales_hour_bin', 'duration_bin', 'price_bin', 'trend_signal'], depth=6, eval_metric='MAE', iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

## Feature Importance

Analyzing which features contribute most to model predictions helps understand what drives the model's decisions.

In [37]:
# Feature importance analysis
cls_importance = cls_model.get_feature_importance()
reg_importance = reg_model.get_feature_importance()

importance_df = pd.DataFrame({
    'feature': features,
    'classifier': cls_importance,
    'regressor': reg_importance
})

importance_df = importance_df.sort_values('classifier', ascending=False)

print("CLASSIFIER Feature Importance (Zero vs Non-Zero Detection):")
print("-" * 60)
for _, row in importance_df.iterrows():
    print(f"  {row['feature']:<20} {row['classifier']:>8.2f}")

print("\n" + "=" * 60)

importance_df = importance_df.sort_values('regressor', ascending=False)

print("REGRESSOR Feature Importance (Quantity Prediction):")
print("-" * 60)
for _, row in importance_df.iterrows():
    print(f"  {row['feature']:<20} {row['regressor']:>8.2f}")

CLASSIFIER Feature Importance (Zero vs Non-Zero Detection):
------------------------------------------------------------
  hist_avg                35.64
  sales_hour_bin          10.31
  route                    9.93
  number_of_passengers     7.30
  price_bin                5.71
  item_id                  4.91
  hist_avg_per_pax         4.60
  duration_bin             3.54
  pax_bin                  3.20
  item_trend_with_threshold     3.03
  am_pm                    2.61
  category                 2.54
  is_outbound              2.43
  flight_duration_hours     1.81
  trend_signal             1.38
  hist_level_used          1.07

REGRESSOR Feature Importance (Quantity Prediction):
------------------------------------------------------------
  hist_avg                57.87
  route                   17.35
  number_of_passengers     4.39
  sales_hour_bin           3.85
  flight_duration_hours     3.09
  item_id                  2.85
  hist_avg_per_pax         2.33
  price_bin           

In [38]:
# predictions with threshold
cls_proba = cls_model.predict_proba(X_test)[:, 1]
threshold = 0.5
cls_pred = (cls_proba >= threshold).astype(int)

reg_pred = reg_model.predict(X_test)
reg_pred = reg_pred.round().clip(0).astype(int)

final_pred = np.where(cls_pred == 0, 0, reg_pred)

In [39]:
# checking different thresholds
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    cls_pred = (cls_proba >= t).astype(int)
    final_pred = np.where(cls_pred == 0, 0, reg_pred)

    mae = mean_absolute_error(y_test, final_pred)
    print(f"\nthreshold={t:.1f} → MAE={mae:.2f}\n")
    evaluate_business_metrics(y_test, final_pred)


threshold=0.3 → MAE=0.59

           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               3732   56.0       0.000000           1.000000
    potential_waste               2205   33.1      -1.369615           0.094328
potential_lost_sale                733   11.0       1.234652           0.213568

accuracy_score (partial credit): 61.4%

threshold=0.4 → MAE=0.52

           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               4210   63.1       0.000000           1.000000
    potential_waste               1483   22.2      -1.538773           0.139240
potential_lost_sale                977   14.6       1.228250           0.137969

accuracy_score (partial credit): 68.2%

threshold=0.5 → MAE=0.49

           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               4432   66.4       0.000000           1.000000
    potential_waste               1076 

In [40]:
# Get CatBoost predictions (threshold=0.7, best performance)
cls_proba_best = cls_model.predict_proba(X_test)[:, 1]
cls_pred_best = (cls_proba_best >= 0.7).astype(int)
reg_pred_best = reg_model.predict(X_test).round().clip(0).astype(int)
catboost_pred = np.where(cls_pred_best == 0, 0, reg_pred_best)

y_true = test_df["sold_quantity"].values
y_true_binary = (y_true > 0).astype(int)

# Dataset composition
zero_mask = (y_true == 0)
nonzero_mask = ~zero_mask

print(f"Dataset Composition:")
print(f"  Zeros: {zero_mask.sum():,} ({zero_mask.sum()/len(y_true)*100:.1f}%)")
print(f"  Non-zeros: {nonzero_mask.sum():,} ({nonzero_mask.sum()/len(y_true)*100:.1f}%)")

print("STAGE 1: CLASSIFICATION PERFORMANCE (Zero vs Non-Zero Detection)")

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Confusion matrix
cm = confusion_matrix(y_true_binary, cls_pred_best)
tn, fp, fn, tp = cm.ravel()

print(f"\nConfusion Matrix:")
print(f"                    Predicted: Zero  |  Predicted: Non-Zero")
print(f"  Actual: Zero      {tn:>6,} (TN)    |  {fp:>6,} (FP)")
print(f"  Actual: Non-Zero  {fn:>6,} (FN)    |  {tp:>6,} (TP)")

print(f"\nClassification Metrics:")
print(f"  True Negatives (correctly predicted zeros):  {tn:,} / {zero_mask.sum():,} = {tn/zero_mask.sum()*100:.1f}%")
print(f"  True Positives (correctly detected sales):   {tp:,} / {nonzero_mask.sum():,} = {tp/nonzero_mask.sum()*100:.1f}%")
print(f"  False Positives (predicted sale, was zero):  {fp:,} ({fp/zero_mask.sum()*100:.1f}% of zeros)")
print(f"  False Negatives (missed sales):              {fn:,} ({fn/nonzero_mask.sum()*100:.1f}% of non-zeros)")

print(f"\nOverall Classification Performance:")
accuracy = (tn + tp) / len(y_true)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"  Accuracy:  {accuracy*100:.1f}%")
print(f"  Precision: {precision*100:.1f}% (when model predicts sale, it's correct)")
print(f"  Recall:    {recall*100:.1f}% (% of actual sales detected)")
print(f"  F1-Score:  {f1:.3f}")
print(f"  ROC-AUC:   {roc_auc_score(y_true_binary, cls_proba_best):.3f}")

print("STAGE 2: REGRESSION PERFORMANCE (Quantity Prediction on Detected Sales)")


# Filter to records where classifier predicted non-zero
predicted_sales_mask = (cls_pred_best == 1)
actual_nonzero_predicted_sale = nonzero_mask & predicted_sales_mask

if actual_nonzero_predicted_sale.sum() > 0:
    print(f"\nRecords where model predicted a sale: {predicted_sales_mask.sum():,}")
    print(f"  Of these, actual non-zeros: {actual_nonzero_predicted_sale.sum():,} ({actual_nonzero_predicted_sale.sum()/predicted_sales_mask.sum()*100:.1f}%)")

    # Regression performance on correctly identified non-zeros
    reg_mae = mean_absolute_error(y_true[actual_nonzero_predicted_sale],
                                    reg_pred_best[actual_nonzero_predicted_sale])
    exact_match = (y_true[actual_nonzero_predicted_sale] == reg_pred_best[actual_nonzero_predicted_sale]).sum()
    within_1 = (np.abs(y_true[actual_nonzero_predicted_sale] - reg_pred_best[actual_nonzero_predicted_sale]) <= 1).sum()
    within_2 = (np.abs(y_true[actual_nonzero_predicted_sale] - reg_pred_best[actual_nonzero_predicted_sale]) <= 2).sum()

    print(f"\nRegression Accuracy (on correctly detected sales):")
    print(f"  Exact match:   {exact_match:,} ({exact_match/actual_nonzero_predicted_sale.sum()*100:.1f}%)")
    print(f"  Within ±1 unit: {within_1:,} ({within_1/actual_nonzero_predicted_sale.sum()*100:.1f}%)")
    print(f"  Within ±2 units: {within_2:,} ({within_2/actual_nonzero_predicted_sale.sum()*100:.1f}%)")
    print(f"  MAE: {reg_mae:.3f} units")

print("\n" + "="*80)
print("OVERALL MODEL PERFORMANCE")
print("="*80)

print(f"\nFinal Predictions (after both stages):")
overall_exact = (catboost_pred == y_true).sum()
overall_mae = mean_absolute_error(y_true, catboost_pred)

# Break down by zero/non-zero
zero_correct = (catboost_pred[zero_mask] == 0).sum()
nonzero_exact = (catboost_pred[nonzero_mask] == y_true[nonzero_mask]).sum()
nonzero_within_1 = (np.abs(catboost_pred[nonzero_mask] - y_true[nonzero_mask]) <= 1).sum()
nonzero_within_2 = (np.abs(catboost_pred[nonzero_mask] - y_true[nonzero_mask]) <= 2).sum()

print(f"\nPerformance on ZEROS ({zero_mask.sum():,} records):")
print(f"  Correctly predicted zero: {zero_correct:,} ({zero_correct/zero_mask.sum()*100:.1f}%)")

print(f"\nPerformance on NON-ZEROS ({nonzero_mask.sum():,} records):")
print(f"  Exact match:        {nonzero_exact:,} ({nonzero_exact/nonzero_mask.sum()*100:.1f}%)")
print(f"  Within ±1 unit:     {nonzero_within_1:,} ({nonzero_within_1/nonzero_mask.sum()*100:.1f}%)")
print(f"  Within ±2 units:    {nonzero_within_2:,} ({nonzero_within_2/nonzero_mask.sum()*100:.1f}%)")
print(f"  MAE: {mean_absolute_error(y_true[nonzero_mask], catboost_pred[nonzero_mask]):.3f}")

print(f"\nOverall:")
print(f"  Exact match accuracy: {overall_exact:,}/{len(y_true):,} = {overall_exact/len(y_true)*100:.1f}%")
print(f"  MAE: {overall_mae:.3f}")

Dataset Composition:
  Zeros: 4,755 (71.3%)
  Non-zeros: 1,915 (28.7%)
STAGE 1: CLASSIFICATION PERFORMANCE (Zero vs Non-Zero Detection)

Confusion Matrix:
                    Predicted: Zero  |  Predicted: Non-Zero
  Actual: Zero       4,553 (TN)    |     202 (FP)
  Actual: Non-Zero   1,449 (FN)    |     466 (TP)

Classification Metrics:
  True Negatives (correctly predicted zeros):  4,553 / 4,755 = 95.8%
  True Positives (correctly detected sales):   466 / 1,915 = 24.3%
  False Positives (predicted sale, was zero):  202 (4.2% of zeros)
  False Negatives (missed sales):              1,449 (75.7% of non-zeros)

Overall Classification Performance:
  Accuracy:  75.2%
  Precision: 69.8% (when model predicts sale, it's correct)
  Recall:    24.3% (% of actual sales detected)
  F1-Score:  0.361
  ROC-AUC:   0.773
STAGE 2: REGRESSION PERFORMANCE (Quantity Prediction on Detected Sales)

Records where model predicted a sale: 668
  Of these, actual non-zeros: 466 (69.8%)

Regression Accuracy (on

## CatBoost variant 2 — SqrtBalanced classifier + Poisson regressor

Second CatBoost setup to see whether class rebalancing (`auto_class_weights='SqrtBalanced'`) improves recall on the classifier stage and whether a Poisson loss on the regressor better matches the count-data distribution of `sold_quantity`.

All evaluation blocks (feature importance, threshold sweep, detailed breakdown) mirror the first variant so that both models can be compared head-to-head.

In [41]:
cls_model_v2 = CatBoostClassifier(
    iterations=600,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features,
    loss_function='Logloss',
    eval_metric='F1',
    auto_class_weights='SqrtBalanced',
    random_seed=42,
    verbose=100
)

cls_model_v2.fit(X_train, y_train_cls)

0:	learn: 0.6380128	total: 16.4ms	remaining: 9.8s
100:	learn: 0.6450303	total: 2.02s	remaining: 9.98s
200:	learn: 0.6499634	total: 3.87s	remaining: 7.69s
300:	learn: 0.6549508	total: 5.77s	remaining: 5.73s
400:	learn: 0.6593236	total: 7.59s	remaining: 3.77s
500:	learn: 0.6631543	total: 9.4s	remaining: 1.86s
599:	learn: 0.6666417	total: 11.2s	remaining: 0us


CatBoostClassifier(auto_class_weights='SqrtBalanced', cat_features=['item_id', 'category', 'is_outbound', 'am_pm', 'route', 'pax_bin', 'sales_hour_bin', 'duration_bin', 'price_bin', 'trend_signal'], depth=6, eval_metric='F1', iterations=600, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [42]:
reg_model_v2 = CatBoostRegressor(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features,
    loss_function='Poisson',
    eval_metric='MAE',
    random_seed=42,
    verbose=100
)
reg_model_v2.fit(X_train_reg, y_train_reg)

0:	learn: 1.7589606	total: 9.28ms	remaining: 2.77s
100:	learn: 1.3308080	total: 1.01s	remaining: 1.98s
200:	learn: 1.3311076	total: 1.92s	remaining: 947ms
299:	learn: 1.3312824	total: 2.83s	remaining: 0us


CatBoostRegressor(cat_features=['item_id', 'category', 'is_outbound', 'am_pm', 'route', 'pax_bin', 'sales_hour_bin', 'duration_bin', 'price_bin', 'trend_signal'], depth=6, eval_metric='MAE', iterations=300, learning_rate=0.05, loss_function='Poisson', random_seed=42, verbose=100)

In [45]:
# Feature importance analysis (v2)
cls_importance_v2 = cls_model_v2.get_feature_importance()
reg_importance_v2 = reg_model_v2.get_feature_importance()

importance_df_v2 = pd.DataFrame({
    'feature': features,
    'classifier': cls_importance_v2,
    'regressor': reg_importance_v2
})

importance_df_v2 = importance_df_v2.sort_values('classifier', ascending=False)

print("CLASSIFIER v2 Feature Importance (SqrtBalanced, Zero vs Non-Zero):")
print("-" * 60)
for _, row in importance_df_v2.iterrows():
    print(f"  {row['feature']:<20} {row['classifier']:>8.2f}")

print("\n" + "=" * 60)

importance_df_v2 = importance_df_v2.sort_values('regressor', ascending=False)

print("REGRESSOR v2 Feature Importance (Poisson loss):")
print("-" * 60)
for _, row in importance_df_v2.iterrows():
    print(f"  {row['feature']:<20} {row['regressor']:>8.2f}")

CLASSIFIER v2 Feature Importance (SqrtBalanced, Zero vs Non-Zero):
------------------------------------------------------------
  hist_avg                31.62
  route                   11.08
  sales_hour_bin          10.34
  number_of_passengers     7.78
  hist_avg_per_pax         5.74
  price_bin                5.53
  item_id                  4.85
  pax_bin                  3.91
  item_trend_with_threshold     3.15
  category                 3.06
  am_pm                    2.65
  duration_bin             2.50
  flight_duration_hours     2.32
  is_outbound              2.31
  trend_signal             2.10
  hist_level_used          1.06

REGRESSOR v2 Feature Importance (Poisson loss):
------------------------------------------------------------
  hist_avg                55.94
  route                    8.96
  item_id                  7.05
  hist_avg_per_pax         6.07
  number_of_passengers     4.03
  duration_bin             3.79
  sales_hour_bin           3.45
  price_bin         

In [46]:
# predictions with threshold (v2)
cls_proba_v2 = cls_model_v2.predict_proba(X_test)[:, 1]

reg_pred_v2 = reg_model_v2.predict(X_test)
reg_pred_v2 = reg_pred_v2.round().clip(0).astype(int)

In [47]:
# checking different thresholds (v2)
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    cls_pred_v2 = (cls_proba_v2 >= t).astype(int)
    final_pred_v2 = np.where(cls_pred_v2 == 0, 0, reg_pred_v2)

    mae = mean_absolute_error(y_test, final_pred_v2)
    print(f"\nthreshold={t:.1f} → MAE={mae:.2f}\n")
    evaluate_business_metrics(y_test, final_pred_v2)


threshold=0.3 → MAE=0.65

           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               3294   49.4       0.000000           1.000000
    potential_waste               2794   41.9      -1.294560           0.076183
potential_lost_sale                582    8.7       1.278351           0.275195

accuracy_score (partial credit): 55.0%

threshold=0.4 → MAE=0.55

           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               3979   59.7       0.000000           1.000000
    potential_waste               1862   27.9      -1.438238           0.114316
potential_lost_sale                829   12.4       1.226779           0.178625

accuracy_score (partial credit): 65.1%

threshold=0.5 → MAE=0.51

           category  number of records  share  avg deviation  relative accuracy
accurate_prediction               4293   64.4       0.000000           1.000000
    potential_waste               1308 

In [48]:
# Detailed evaluation for v2 (threshold=0.7 chosen to match v1 selection)
cls_proba_best_v2 = cls_model_v2.predict_proba(X_test)[:, 1]
cls_pred_best_v2 = (cls_proba_best_v2 >= 0.7).astype(int)
reg_pred_best_v2 = reg_model_v2.predict(X_test).round().clip(0).astype(int)
catboost_pred_v2 = np.where(cls_pred_best_v2 == 0, 0, reg_pred_best_v2)

y_true = test_df["sold_quantity"].values
y_true_binary = (y_true > 0).astype(int)

zero_mask = (y_true == 0)
nonzero_mask = ~zero_mask

print(f"Dataset Composition:")
print(f"  Zeros: {zero_mask.sum():,} ({zero_mask.sum()/len(y_true)*100:.1f}%)")
print(f"  Non-zeros: {nonzero_mask.sum():,} ({nonzero_mask.sum()/len(y_true)*100:.1f}%)")

print("STAGE 1: CLASSIFICATION PERFORMANCE (v2 — SqrtBalanced)")

cm_v2 = confusion_matrix(y_true_binary, cls_pred_best_v2)
tn, fp, fn, tp = cm_v2.ravel()

print(f"\nConfusion Matrix:")
print(f"                    Predicted: Zero  |  Predicted: Non-Zero")
print(f"  Actual: Zero      {tn:>6,} (TN)    |  {fp:>6,} (FP)")
print(f"  Actual: Non-Zero  {fn:>6,} (FN)    |  {tp:>6,} (TP)")

print(f"\nClassification Metrics:")
print(f"  True Negatives (correctly predicted zeros):  {tn:,} / {zero_mask.sum():,} = {tn/zero_mask.sum()*100:.1f}%")
print(f"  True Positives (correctly detected sales):   {tp:,} / {nonzero_mask.sum():,} = {tp/nonzero_mask.sum()*100:.1f}%")
print(f"  False Positives (predicted sale, was zero):  {fp:,} ({fp/zero_mask.sum()*100:.1f}% of zeros)")
print(f"  False Negatives (missed sales):              {fn:,} ({fn/nonzero_mask.sum()*100:.1f}% of non-zeros)")

print(f"\nOverall Classification Performance:")
accuracy = (tn + tp) / len(y_true)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"  Accuracy:  {accuracy*100:.1f}%")
print(f"  Precision: {precision*100:.1f}%")
print(f"  Recall:    {recall*100:.1f}%")
print(f"  F1-Score:  {f1:.3f}")
print(f"  ROC-AUC:   {roc_auc_score(y_true_binary, cls_proba_best_v2):.3f}")

print("STAGE 2: REGRESSION PERFORMANCE (v2 — Poisson)")

predicted_sales_mask_v2 = (cls_pred_best_v2 == 1)
actual_nonzero_predicted_sale_v2 = nonzero_mask & predicted_sales_mask_v2

if actual_nonzero_predicted_sale_v2.sum() > 0:
    print(f"\nRecords where model predicted a sale: {predicted_sales_mask_v2.sum():,}")
    print(f"  Of these, actual non-zeros: {actual_nonzero_predicted_sale_v2.sum():,} ({actual_nonzero_predicted_sale_v2.sum()/predicted_sales_mask_v2.sum()*100:.1f}%)")

    reg_mae_v2 = mean_absolute_error(
        y_true[actual_nonzero_predicted_sale_v2],
        reg_pred_best_v2[actual_nonzero_predicted_sale_v2]
    )
    exact_match_v2 = (y_true[actual_nonzero_predicted_sale_v2] == reg_pred_best_v2[actual_nonzero_predicted_sale_v2]).sum()
    within_1_v2 = (np.abs(y_true[actual_nonzero_predicted_sale_v2] - reg_pred_best_v2[actual_nonzero_predicted_sale_v2]) <= 1).sum()
    within_2_v2 = (np.abs(y_true[actual_nonzero_predicted_sale_v2] - reg_pred_best_v2[actual_nonzero_predicted_sale_v2]) <= 2).sum()

    print(f"\nRegression Accuracy (on correctly detected sales):")
    print(f"  Exact match:   {exact_match_v2:,} ({exact_match_v2/actual_nonzero_predicted_sale_v2.sum()*100:.1f}%)")
    print(f"  Within ±1 unit: {within_1_v2:,} ({within_1_v2/actual_nonzero_predicted_sale_v2.sum()*100:.1f}%)")
    print(f"  Within ±2 units: {within_2_v2:,} ({within_2_v2/actual_nonzero_predicted_sale_v2.sum()*100:.1f}%)")
    print(f"  MAE: {reg_mae_v2:.3f} units")

print("\n" + "="*80)
print("OVERALL MODEL PERFORMANCE (v2)")
print("="*80)

overall_exact_v2 = (catboost_pred_v2 == y_true).sum()
overall_mae_v2 = mean_absolute_error(y_true, catboost_pred_v2)

zero_correct_v2 = (catboost_pred_v2[zero_mask] == 0).sum()
nonzero_exact_v2 = (catboost_pred_v2[nonzero_mask] == y_true[nonzero_mask]).sum()
nonzero_within_1_v2 = (np.abs(catboost_pred_v2[nonzero_mask] - y_true[nonzero_mask]) <= 1).sum()
nonzero_within_2_v2 = (np.abs(catboost_pred_v2[nonzero_mask] - y_true[nonzero_mask]) <= 2).sum()

print(f"\nPerformance on ZEROS ({zero_mask.sum():,} records):")
print(f"  Correctly predicted zero: {zero_correct_v2:,} ({zero_correct_v2/zero_mask.sum()*100:.1f}%)")

print(f"\nPerformance on NON-ZEROS ({nonzero_mask.sum():,} records):")
print(f"  Exact match:        {nonzero_exact_v2:,} ({nonzero_exact_v2/nonzero_mask.sum()*100:.1f}%)")
print(f"  Within ±1 unit:     {nonzero_within_1_v2:,} ({nonzero_within_1_v2/nonzero_mask.sum()*100:.1f}%)")
print(f"  Within ±2 units:    {nonzero_within_2_v2:,} ({nonzero_within_2_v2/nonzero_mask.sum()*100:.1f}%)")
print(f"  MAE: {mean_absolute_error(y_true[nonzero_mask], catboost_pred_v2[nonzero_mask]):.3f}")

print(f"\nOverall:")
print(f"  Exact match accuracy: {overall_exact_v2:,}/{len(y_true):,} = {overall_exact_v2/len(y_true)*100:.1f}%")
print(f"  MAE: {overall_mae_v2:.3f}")

Dataset Composition:
  Zeros: 4,755 (71.3%)
  Non-zeros: 1,915 (28.7%)
STAGE 1: CLASSIFICATION PERFORMANCE (v2 — SqrtBalanced)

Confusion Matrix:
                    Predicted: Zero  |  Predicted: Non-Zero
  Actual: Zero       4,459 (TN)    |     296 (FP)
  Actual: Non-Zero   1,318 (FN)    |     597 (TP)

Classification Metrics:
  True Negatives (correctly predicted zeros):  4,459 / 4,755 = 93.8%
  True Positives (correctly detected sales):   597 / 1,915 = 31.2%
  False Positives (predicted sale, was zero):  296 (6.2% of zeros)
  False Negatives (missed sales):              1,318 (68.8% of non-zeros)

Overall Classification Performance:
  Accuracy:  75.8%
  Precision: 66.9%
  Recall:    31.2%
  F1-Score:  0.425
  ROC-AUC:   0.773
STAGE 2: REGRESSION PERFORMANCE (v2 — Poisson)

Records where model predicted a sale: 893
  Of these, actual non-zeros: 597 (66.9%)

Regression Accuracy (on correctly detected sales):
  Exact match:   173 (29.0%)
  Within ±1 unit: 457 (76.5%)
  Within ±2 units

In [49]:
# Head-to-head comparison v1 vs v2 at threshold=0.7
comparison = pd.DataFrame({
    'metric': ['MAE', 'Exact-match accuracy', 'Non-zero exact match', 'Non-zero within ±1', 'Non-zero within ±2'],
    'v1 (Logloss + RMSE)': [
        f"{mean_absolute_error(y_true, catboost_pred):.3f}",
        f"{(catboost_pred == y_true).sum() / len(y_true) * 100:.1f}%",
        f"{(catboost_pred[nonzero_mask] == y_true[nonzero_mask]).sum() / nonzero_mask.sum() * 100:.1f}%",
        f"{(np.abs(catboost_pred[nonzero_mask] - y_true[nonzero_mask]) <= 1).sum() / nonzero_mask.sum() * 100:.1f}%",
        f"{(np.abs(catboost_pred[nonzero_mask] - y_true[nonzero_mask]) <= 2).sum() / nonzero_mask.sum() * 100:.1f}%",
    ],
    'v2 (SqrtBalanced + Poisson)': [
        f"{mean_absolute_error(y_true, catboost_pred_v2):.3f}",
        f"{(catboost_pred_v2 == y_true).sum() / len(y_true) * 100:.1f}%",
        f"{(catboost_pred_v2[nonzero_mask] == y_true[nonzero_mask]).sum() / nonzero_mask.sum() * 100:.1f}%",
        f"{(np.abs(catboost_pred_v2[nonzero_mask] - y_true[nonzero_mask]) <= 1).sum() / nonzero_mask.sum() * 100:.1f}%",
        f"{(np.abs(catboost_pred_v2[nonzero_mask] - y_true[nonzero_mask]) <= 2).sum() / nonzero_mask.sum() * 100:.1f}%",
    ]
})
print(comparison.to_string(index=False))

              metric v1 (Logloss + RMSE) v2 (SqrtBalanced + Poisson)
                 MAE               0.448                       0.457
Exact-match accuracy               70.2%                       69.4%
Non-zero exact match                6.6%                        9.0%
  Non-zero within ±1               76.1%                       79.0%
  Non-zero within ±2               92.3%                       93.3%


## Missed sales analysis — v1

Since **v1 (Logloss + RMSE)** shows the most balanced performance, we take a closer look at where it under-predicts.

For each item we report:
- `lost_sale_count` — number of records where actual > predicted
- `share_of_item_records` — how frequently this item is under-predicted (as % of item's test records)
- `avg_miss` — average size of the gap `actual - predicted` on missed records
- `max_miss` — worst single-record gap
- `miss_range` — bucketed distribution of the gap size (`1`, `2`, `3-5`, `6+`)

In [50]:
# Missed sales breakdown per item (v1)
diff = test_df['sold_quantity'].values - catboost_pred
lost_mask = diff > 0

lost_df = test_df.loc[lost_mask, ['item_id']].copy()
lost_df['miss'] = diff[lost_mask]

item_records = test_df.groupby('item_id').size().rename('total_records')

per_item = lost_df.groupby('item_id').agg(
    lost_sale_count=('miss', 'size'),
    avg_miss=('miss', 'mean'),
    max_miss=('miss', 'max'),
    total_missed_units=('miss', 'sum'),
).join(item_records, how='left')

per_item['share_of_item_records'] = (per_item['lost_sale_count'] / per_item['total_records'] * 100).round(1)
per_item['avg_miss'] = per_item['avg_miss'].round(2)

per_item = per_item[[
    'lost_sale_count',
    'total_records',
    'share_of_item_records',
    'avg_miss',
    'max_miss',
    'total_missed_units',
]].sort_values('lost_sale_count', ascending=False)

print("Top items by frequency of under-prediction (v1):")
print(per_item.to_string())

Top items by frequency of under-prediction (v1):
          lost_sale_count  total_records  share_of_item_records  avg_miss  max_miss  total_missed_units
item_id                                                                                                
PROD_094              201            695                   28.9      1.24       4.0               249.0
PROD_093              187            695                   26.9      1.22       3.0               229.0
PROD_105              171            695                   24.6      1.41       5.0               241.0
PROD_096              168            695                   24.2      1.24       3.0               208.0
PROD_200              167            639                   26.1      1.32       4.0               221.0
PROD_216              165            695                   23.7      1.19       3.0               197.0
PROD_199              152            639                   23.8      1.95       8.0               296.0
PROD_210       

In [51]:
# Miss-size distribution per item (v1)
bins = [0, 1, 2, 5, np.inf]
labels = ['1', '2', '3-5', '6+']
lost_df['miss_bucket'] = pd.cut(lost_df['miss'], bins=bins, labels=labels, right=True)

miss_range = (
    lost_df
    .groupby(['item_id', 'miss_bucket'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=labels, fill_value=0)
)

miss_range['total_misses'] = miss_range.sum(axis=1)
miss_range = miss_range.sort_values('total_misses', ascending=False)

print("Distribution of miss size per item (v1):")
print(miss_range.to_string())

Distribution of miss size per item (v1):
miss_bucket    1   2  3-5  6+  total_misses
item_id                                    
PROD_094     162  31    8   0           201
PROD_093     151  30    6   0           187
PROD_105     124  32   15   0           171
PROD_096     136  24    8   0           168
PROD_200     121  40    6   0           167
PROD_216     138  22    5   0           165
PROD_199      79  27   45   1           152
PROD_210     114  24    7   0           145
PROD_209      93  22    3   0           118
PROD_118      73   8    0   0            81
